In [1]:
import numpy as np
from numpy import random
import itertools

In [2]:
class NK_model(object):
    
    def __init__(self, n, k):
        self.n = n
        self.k = k
        self.start_type = 0    # the initial given start genotype
        self.get_epistic_site()  # get the epistic network among the sites (including the site itself) 
        
        self.allelic_each_site = [[] for _ in range(self.n)]   # will be used to store the episitc combinations at each site
        self.fitness_list = [{} for _ in range(self.n)]   # will be used to store the fitness of epistic combinations at each
                                                          # site
        self.genotype_list = []   # will be used to store the genotypes generated (subset of all the genotypes)
        self.geno_fitness = []   # will be used to store the calculated fitness of genotypes generated
        self.landscape = {}  # will be used to store the generated landscape (subset of the whole landscape)
        
        
        
    @staticmethod
    def generate_start_type(seq_length):
        '''generate a genotype as the start_type
        
        parameters: 
        ----------
            seq_length: int, the length of the wanted genotype sequence
        output: 
        -------
            str that represents the start_type
        '''
        
        a_list = []
        for i in range(seq_length):  # generate a list randomly composed of 0 and 1
            a_list.append(random.randint(0, 2))
            
        start_type_list = []   
        for i in a_list:   # convert the integer elements (0 and 1) in a_list into string elements ('0' and '1') and append 
            start_type_list.append(str(i))   # into start_type_list
            
        start_type = ''.join(start_type_list)  # join the elements in start_type_list into a string
        
        return start_type
    
    
    @staticmethod
    def creat_neighbor(genotype, mutant_num):
        '''creat the mutant_num-mutant neighbors of the given genotype
        
        parameters: 
        ----------
            genotype: str that repesents genotype, eg: "01", "00"
            mutant_num: int, the number of mutant site to creat neighbors
            
        output: 
        -------
            list that store the mutant_num-mutant neighbors of the given genotype
        '''
        
        total_neigh_list= []  # will be used to store the created neighbors of the given genotype
    
        geno_list=list(genotype) # convert the given genotype (str) into a list
    
        indices = list(range(len(geno_list))) # get all the indices for the given genotype
    
        mutant_indices = list(itertools.combinations(indices, mutant_num)) # get all possible combinations of picking 
                                                                    # mutant_num intergers from indices (i.e. mutant sites)
    
        for i in mutant_indices:  # get all its mutant_num -mutant neighbors of the given genotype
            neigh_list = geno_list.copy()
            for j in i:
                if geno_list[j] == '0':
                    neigh_list[j] = '1'
                else:
                    neigh_list[j] = '0'
            
            total_neigh_list.append(''.join(neigh_list))
    
        return total_neigh_list
    
    
    def get_epistic_site(self):
        '''Generate the epistic networks among the sites'''

        self.epistic_site = []
        for i in range(self.n):
            all_site = [x for x in range(self.n) if x != i]  # get the indexes of all sites excluding i
            
            indices = set()   # will be used to store the indexes of epistic sites
            while len(indices) < self.k:
                indices.add(random.randint(0, len(all_site))) # randomly pick the indexes of k elements from all_site

            indices = list(indices)  # convert the set into list
            
            one_site = [i]   # first append the site under study into one_site (a little bit different from 0615 code)
            for j in indices:  # get the epistic sites of site i
                one_site.append(all_site[j])
            
            self.epistic_site.append(one_site) 
           
        
    @staticmethod     
    def get_epistic_combination(genotype, network):
        '''generate the epistic pairs in a given genotype according to the network
        
        parameters: 
        ----------
            genotype: str that repesents a genotype, 
            network: nested list that stores the epistic networks among the sites (self.epistic_site)
            
        output: 
        -------
            list that stores all the epistic combinations in the given genotype
        '''
        
        epistic_combination = []   # will be used to store the epistic combinations in the given genotype
        
        for j in range(len(genotype)):
            one_site = []   
            for i in network[j]:   # get all epistic alleles in one site (store in list one_site)
                one_site.append(genotype[i])
                
            epistic_combination.append(''.join(one_site))  # combine the epistic alleles and store into epistic_combination
                
        return epistic_combination
        
        
    @staticmethod      
    def cal_fitness(epistic_pair, fitness_list):
        '''calculate the fitness of a given genotype according to the fitness_list (here the epistic_pair is generate by
        the genotype, i.e. the epistic_combination in get_epistic_combination(genotype, network) method)
        
        parameters: 
        ----------
            epistic_pair: list that stores the epistic combinations in a genotype  
            fitness_list: list that stores the dictionary which indicates the fitness of each epistic combinations 
                          in each site
            
        output: 
        -------
            float, the calculated fitness of a genotype (the genotype which corresponded to epistic_pair)
        '''
        
        one_fitness = []    
        
        for i in range(len(epistic_pair)):  # fitness_list[i] is the fitness dictionary that corresponds to site i.  
            one_fitness.append(fitness_list[i][epistic_pair[i]])  # epistic_pair[i] is the epistic combination in site i (the
                                                                # key in the dictionary)
        i_fitness = sum(one_fitness)/len(one_fitness)  # calculate the fitness
            
        return i_fitness
          
        
    @staticmethod
    def whether_fixed(fitness_1, fitness_2):
        ''' check if the possible_next_type in an adaptive walk can be fixed
        
        parameters: 
        ----------
            fitness_1: float, the fitness of the previous genotype  
            fitness_2: float, the fitness of the possible_next_type
            
        output: 
        -------
            boolean, indicating if the possible_next_type can be fixed (True: can; False: cannot)
        '''
        
        s = (fitness_2/fitness_1)-1  # calculate the selective coefficent
        fix_pro = 2*s  # calculate the possibiliity of fixation
        
        random_no = random.random(1)[0]  # generate a random number from uniform distrubtion
        
        if fix_pro > random_no:  # if fix_pro is larger than random_no, return True (can be  fixed)
            return True
        else:          # else cannot be fixed
            return False
    
    
    
    def adaptive_walk(self):
        '''Simulate one step  of adaptive walk along the landscape'''
        
        if self.start_type==0:  # self.start_type==0 (the initial given value), means initial step, randomly generates a 
                                # genotype as start_type (length is self.n)
            self.start_type = self.generate_start_type(self.n)
        else:  # start!='', not initial step, just pass
            pass
        
        neighbor_list = [self.start_type] # this list will be used to store the self.start_type and all its neighbors
                                        # (next step will select from the neighbors of self.start_type)
        
        neighbor_list += self.creat_neighbor(self.start_type, 1) # get all 1-mutant neighbors of start_type
                
        all_epistic_pair = [] # this list will be used to store all epistic combinations for genotypes in neighbor_list
        
        for i in neighbor_list:  # i is genotype in neighbor_list, self.epistic_site is network among the site
            all_epistic_pair.append(self.get_epistic_combination(i, self.epistic_site))
        
        
        allelic_set = []  # list that will be used to store all the epistic combinations in site 0, 1, 2, ..., self.n-1 
                        # (use set to exclude the repeat ones) for genotypes in neighbor_list
        
        for j in range(self.n): 
            pos_allelic_pair = [] # will be used to store all epistic combinations in site j for genotypes in neighbor_list
            for s in all_epistic_pair:
                pos_allelic_pair.append(s[j])   
            
            pos_allelic_set = list(set(pos_allelic_pair))  # convert the pos_allelic_pair to set then convert back to list to
                                                    # get the non-repeated epistic combinations in site j
                
            allelic_set.append(pos_allelic_set)  # append the non-repeated epistic combinations in site j into allelic_set to
                                        # get epistic combinations in all sites
            
        for s in range(self.n): # the length of allelic_set is self.n
            for i in allelic_set[s]: # i is the allelic combinations in site s, self.allelic_each_site is used to store the 
                                    #  episitc combinations at each site, self.allelic_each_site[s] is episitc combinations 
                                    # at site s that has already been generated
                if i not in self.allelic_each_site[s]: # if i is not in self.allelic_each_site[s], means epistic combination i
                                 # has not been generated, just append i into self.allelic_each_site[s] and generate a fitness
                                 # for i and store into self.fitness_list[s](i will be the key)
                    self.allelic_each_site[s].append(i)
                    self.fitness_list[s][i] = random.random(1)[0]
            
        w_fitness = []   # will be used to store the calculated fitness of each genotype in neighbor_list
        
        for i in neighbor_list:
            if i not in self.genotype_list: # i not in self.genotype_list, means the fitness of genotype i has not been
                                            # calculated--> need to be calculated this time
                index_i = neighbor_list.index(i) # first get the index of i (this index is the same as the epistic 
                                                # combinations for genotype i in all_epistic_pair)
                i_fitness = self.cal_fitness(all_epistic_pair[index_i], self.fitness_list)
                w_fitness.append(i_fitness)
            else:  # otherwise it means that the fitness of genotype i has been calculated, just find it from the landscape
                w_fitness.append(self.landscape[i])
            
        neigh_fitness = dict(zip(neighbor_list[1:], w_fitness[1:]))  # creat a dictionary to store the genotypes and fitness 
                                                            # of the 1-mutant neighbors of self.start_type (start_type is not 
                                                            # included as it won't be selected for self.next_step) 
                                                            # key: genotype; value: fitness
    
        self.start_type_fitness = w_fitness[0]  # get the fitness of self.start_type
        
        possible_neigh = {k : v for k, v in neigh_fitness.items() if v >= self.start_type_fitness}  # get all possible   
                                                                    # neighbors for self.next_step (only if the fitness >= 
                                                                    # self.start_type fitness can be possible neighbors 
                                                                    # for self.next_step) 
        
        if possible_neigh:   # if possible_neigh is not empty, it means that possible genotypes for self.next_step exist;
                             # select the genotype for self.next_step according to their fitness
                
            pop_gens = list(possible_neigh.keys())   # get all the possible genotypes of self.next_step
            possible_next_type = pop_gens[random.randint(0, len(pop_gens))]  # find the possible_next_type and 
            possible_next_type_fit = possible_neigh[possible_next_type] # possible_next_type_fitness
            
            if self.whether_fixed(self.start_type_fitness, possible_next_type_fit): # check if the next_type can be fixed
                self.next_type = possible_next_type         # if can be fixed, then updated the self.next_type and its fitness
                self.next_type_fitness = possible_next_type_fit
            else:   # if possible_next_type cannot be fixed, then just return to the start_type and do the adaptive walk again
                self.next_type = self.start_type
                self.next_type_fitness = self.start_type_fitness
                                                                                
        else: # if possible_neigh is empty, it means that there if no option for self.next_step-->cannot evolve
              #-->reach local optima
            self.next_type = ''
            self.next_type_fitness = 0   
           
    
        self.genotype_list.extend(neighbor_list)  # updating the self.genotype_list by storing all the genotype in 
                                                # neighbor_list into self.genotype_list
            
        self.genotype_list = list(set(self.genotype_list))  # exclude the repeating genotypes in self.genotype_list
        
        landscape_subset = dict(zip(neighbor_list, w_fitness)) # the subset of landscape generated in this adaptive_walk step
        
        self.landscape.update(landscape_subset)  # update self.landscape by the subset of landscape generated in this adaptive
                                                 # _walk step
                
        
    def repeat_adaptive_walk(self):
        '''Simulate the random walk in the landscape'''
        
        self.walk_list = []  # this list will be used to store the steps of adaptive walk
        
        fit = self.adaptive_walk()  # run the adaptive_walk method
        
        self.walk_list.append([self.start_type, self.start_type_fitness])  # the initial step
        
        while self.next_type and self.next_type_fitness:  # this means that while next_step!='' and fitness[next_step]!=0.0,  
                                                         #not reach local optima
            self.walk_list.append([self.next_type, self.next_type_fitness]) # the next_step
            self.start_type = self.next_type  # update the self.start_type and self.start_type_fitness
            self.start_type_fitness = self.next_type_fitness
            fit = self.adaptive_walk() # run the random_walk method with the updated self.start_type 
        
        self.walk_genotype = list(set([i[0] for i in self.walk_list]))#get all the genotypes in the walk_list (non-repeated one)
        self.new_walk_list = [[j, self.landscape[j]] for j in self.walk_genotype] # new_walk_list: exclude the repeated genotype
        self.new_walk_list.sort(key=lambda x: x[1])#in self.walk_list and sort according to the fitness (get the true 
                                                  # order of genotypes in an adaptive walk
        self.walk_genotype = [i[0] for i in self.new_walk_list] # update the self.walk_genotype to the true order of genotypes
                                                            # that have been fixed in an adaptive walk
        if len(self.new_walk_list) > 6:   # if the length of the adaptive walk is more than 5 steps (6 genotypes involved)
            self.walk_genotype = self.walk_genotype[:6]  # then just get the first 5 steps to study
            self.new_walk_list = self.new_walk_list[:6]
    
    
    def search_different_allele(self):
        '''check all the alleles in each site for all genotypes involved in an adaptive walk(e.g. site 1: have '0' and '1',
        (means some genotypes have '0' in site 1 and some other have '1' in site 1),site 2: only have '0' (means all genotypes 
        involved in an adaptive walk have '0' in site 2))'''
        
        walk_geno_list = []  
        for i in self.walk_genotype:  # first convert the genotypes into lists (each genotype will have a list) and store in
            walk_geno_list.append(list(i)) # walk_geno_list
        
        
        self.total_allele_list = []  # will be used to store the alleles present in each site for all genotypes involved in 
                                    # an adaptive walk
        for i in range(self.n): # for each site
            allele_list = []  # will be used to store all alleles in site i
            for j in walk_geno_list:
                allele_list.append(j[i])
            
            allele_list = list(set(allele_list))#convert allele_list into a set to exclude the repeated one and then back to list
            self.total_allele_list.append(allele_list) # append the alleles in site i into self.total_allele_list
            
        self.shared_allele_indice = []#will be used to store the indexes of shared alleles for the genotypes in an adaptive walk
        self.diff_allele_indice = []#will be used to store the indexes of different alleles for the genotypes in an adaptive walk
        
        for i in range(self.n):
            if len(self.total_allele_list[i])>1: # this means site i has more than one alleles-->different allele,append i into
                self.diff_allele_indice.append(i) # self.diff_allele_indice
            else: # else means site i only have one allele for all genotypes--> shared allele, append i into self.shared_allele
                self.shared_allele_indice.append(i)   # _indice       

        self.shared_seq = '' # will be used to store the shared sequences for all genotypes
        
        for i in self.shared_allele_indice:
            self.shared_seq+=self.walk_genotype[0][i]  # get the shared sequence by all genotype (use the first genotype in 
                                                      # adaptive walk to construct, also can use other genotypes in adatptive 
                                                      # walk because this sequence is shared by all genotypes
        self.landscape_subset = {}   # this is what we would to get, the subset of landscape which only contain the information
                                    # of different alleles
        for i in self.walk_genotype: # get different allele sequence for each genotype in an adaptive walk
            s = ''
            for j in self.diff_allele_indice:
                s+=i[j]        # the different allele sequence in genotype i
           
            self.landscape_subset[s] = self.landscape[i]  # update the subset of landscape (key is the different allele 
                                                         # sequence in genotype i, value is the fitness of genotype i)
            
            
    def creat_landscape_subset(self):
        '''Generate the subset of lanscape which contains all possible combinations in different allele sites'''
        
        all_possible_combination = [''.join(i) for i in itertools.product('01', repeat = len(self.diff_allele_indice))]
        # generate all possible allele combinations in different allele sites
        
        already_find = list(self.landscape_subset.keys()) # this is allele combinations that have already been got
        
        need_to_find_com = []  # will be used to store the allele combinations that haven't been got
        
        for i in all_possible_combination:
            if i not in already_find:  # get all allele combinations that haven't been got
                need_to_find_com.append(i)
                
        need_to_find_com_list = []
        for i in need_to_find_com:  # convert each of the allele combinations in need_to_find_com to list and 
            need_to_find_com_list.append(list(i)) # store in need_to_find_com_list
            
        self.shared_seq_list = list(self.shared_seq) # convert the shared sequence (str) into a list
        
        orginal_genotype = []   # will be used to store the whole genotype constructed by the shared sequence and different 
                                # allele combinations (will be then used to find the fitness in self.landscape)
        for i in need_to_find_com_list: # only the different allele combinations that haven't been found (i.e. in need_to_find
            for j in self.shared_allele_indice: # _com_list)  will be converted back to the whole genotype sequence
                index_j = self.shared_allele_indice.index(j) 
                i.insert(j, self.shared_seq_list[index_j]) # insert the alleles in shared_seq according to i according to the 
            orginal_genotype.append(''.join(i))  # indices of shared_sequence in whole genotype sequence
            
        orginal_genotype_fitness = [] #will be used to store the fitness of each construced whole genotypes in orginal_genotype
        
        all_geno = list(self.landscape.keys()) # first get all whole genotype sequences in self.landscape
        
        for i in orginal_genotype:
            if i in all_geno:  # i in all_geno--> i already in self.landcape, get the fitness of i directly
                orginal_genotype_fitness.append(self.landscape[i])
            else: # else i not in all_geno--> i not in self.landscape, need to calculate the fitness of i
                allelic_comb_i = self.get_epistic_combination(i, self.epistic_site) # generate the epistic pairs in i 
                                                                     # according to the network (i.e. self.epistic_site)
               
                for j in range(self.n): # check whether the all_comb_i[j] have already been generated and given a fitness in 
                                        # site j
                    if allelic_comb_i[j] not in self.allelic_each_site[j]: # if not, just append the new combination into self.
                        self.allelic_each_site[j].append(allelic_comb_i[j]) # allelic_each_site[j] and given a fitness
                        self.fitness_list[j][allelic_comb_i[j]] = random.random(1)[0]
                
                fitness_i = self.cal_fitness(allelic_comb_i, self.fitness_list) # calcualte the fitness of genotype i
                
                orginal_genotype_fitness.append(fitness_i) # append the fitness of genotype i into orginal_genotype_fitness
                self.landscape[i] = fitness_i # also update self.landscape with genotype i
        
        updated_landscape_subset = dict(zip(need_to_find_com, orginal_genotype_fitness)) # the updated_landscape_subset 
                                    # (need_to_find_com stores the keys, orginal_genotype_fitness stores the fitness,
                                    # need_to_find_com has the same order with orginal_genotype, orginal_genotype has the 
                                    # same order with orginal_genotype_fitness, so need_to_find_com has the same order with
                                    # with orginal_genotype_fitness)
                        
        self.landscape_subset.update(updated_landscape_subset) # updated the self.landscape_subset wuth updated_landscape_subset.

In [16]:
a = NK_model(6, 3)
print(a.epistic_site)

a.repeat_adaptive_walk()
print('1', a.walk_list)
print('2', a.walk_genotype)
print('3', a.new_walk_list)
print('LAND', a.landscape, len(list(a.landscape.keys())))

a.search_different_allele()
print('5', a.total_allele_list)
print('6', a.diff_allele_indice)
print('7', a.shared_seq)
print('8', a.landscape_subset)

a.creat_landscape_subset()
print('9', a.landscape_subset, len(list(a.landscape_subset.keys())))
print('UPDATED LAND', a.landscape, len(list(a.landscape.keys())))

[[0, 2, 3, 4], [1, 3, 4, 5], [2, 0, 1, 5], [3, 0, 2, 4], [4, 0, 1, 3], [5, 0, 1, 3]]
1 [['000101', 0.46919795717192708], ['000101', 0.46919795717192708], ['000101', 0.46919795717192708], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010101', 0.59775135733837437], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966646705603], ['010100', 0.64166966